# WAHub: Script Scraper & Validator Link Grup WhatsApp Mahasiswa Indonesia
Script ini mencari, mengekstrak, dan memvalidasi link grup WhatsApp untuk mahasiswa Indonesia
dari berbagai universitas, fakultas, dan angkatan tahun 2023-2026.
Script menggunakan pencarian Google dan DuckDuckGo secara gratis (tanpa API key).

In [48]:
import os
import re
import csv
import time
import random
import asyncio
import datetime
from dataclasses import dataclass
from typing import List, Dict, Set, Optional, Tuple
import functools

import functools
import logging

# Sembunyikan pesan WARNING urllib3 saat melakukan Auto-Retry agar terminal tetap bersih
logging.getLogger("urllib3").setLevel(logging.ERROR)

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import aiohttp
try:
    from aiohttp_socks import ProxyConnector
except ImportError:
    ProxyConnector = None
import pandas as pd
from PIL import Image
import io
try:
    from pyzbar.pyzbar import decode as qr_decode
except ImportError:
    qr_decode = None
try:
    import pytesseract
except ImportError:
    pytesseract = None
from bs4 import BeautifulSoup
import threading

# Konfigurasi HTTP Session dengan Auto-Retry (Bypass error saat Tor merestart)
http_session = requests.Session()
retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=3,
    status_forcelist=[429, 500, 502, 503, 504]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
http_session.mount("https://", adapter)
http_session.mount("http://", adapter)

tor_lock = threading.Lock()

## Konfigurasi & Data Referensi
Di sini kita mendefinisikan target jumlah link, daftar tahun angkatan, 
daftar universitas, jurusan, dan template pencarian.

In [49]:
TARGET_VALID_LINKS = float('inf')
CONCURRENT_WORKERS = 400 # Ditingkatkan ke batas absolut sistem (Hati-hati OOM)
TAHUN_RANGE = [2023, 2024, 2025, 2026]

# Konfigurasi Tor Proxy Multiplexing
USE_TOR_PROXY = True
NUM_TOR_INSTANCES = 50
import os

try:
    import aiohttp_socks
except ImportError:
    pass

# Hapus environment variable global agar requests tidak pakai proxy default
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

# Menghindari bentrok port: ControlPort = SocksPort + NUM_TOR_INSTANCES
proxy_pool = [{"socks_port": 9050 + i, "control_port": 9050 + NUM_TOR_INSTANCES + i, "blocked_count": 0, "lock": threading.Lock()} for i in range(NUM_TOR_INSTANCES)]



# Kita akan menggunakan CSV untuk memuat daftar universitas secara dinamis.
# Jika gagal, ini akan menjadi fallback.
DAFTAR_UNIVERSITAS_FALLBACK = [
    "UI", "Universitas Indonesia", "UGM", "Universitas Gadjah Mada", "ITB", "Institut Teknologi Bandung",
    "IPB", "Institut Pertanian Bogor", "Unair", "Universitas Airlangga", "Undip", "Universitas Diponegoro",
    "Unpad", "Universitas Padjadjaran", "ITS", "Institut Teknologi Sepuluh Nopember", "UNS", "Universitas Sebelas Maret",
    "UB", "Universitas Brawijaya", "UNJ", "Universitas Negeri Jakarta", "Unhas", "Universitas Hasanuddin"
]

# Diperluas sesuai permintaan agar lebih komprehensif (termasuk Fakultas dan Jurusan tambahan)
DAFTAR_FAKULTAS_JURUSAN = [
    # --- FAKULTAS ---
    "Fakultas Teknik", "Fakultas Ilmu Komputer", "Fakultas Kedokteran", "Fakultas Ekonomi dan Bisnis",
    "Fakultas Hukum", "Fakultas Ilmu Sosial dan Ilmu Politik", "Fakultas Psikologi",
    "Fakultas Ilmu Pengetahuan Budaya", "Fakultas Ilmu Budaya", "Fakultas Pertanian",
    "Fakultas Peternakan", "Fakultas Kehutanan", "Fakultas MIPA", "Fakultas Farmasi",
    "Fakultas Kesehatan Masyarakat", "Fakultas Kedokteran Gigi", "Fakultas Ilmu Keperawatan",
    "Fakultas Ilmu Komunikasi", "Fakultas Keguruan dan Ilmu Pendidikan", "Fakultas Tarbiyah",
    "Fakultas Syariah", "Fakultas Ushuluddin", "Fakultas Dakwah dan Komunikasi", "Fakultas Adab",
    # --- JURUSAN TEKNIK & KOMPUTER ---
    "Teknik Informatika", "Sistem Informasi", "Ilmu Komputer", "Teknik Elektro", 
    "Teknik Mesin", "Teknik Sipil", "Teknik Industri", "Arsitektur", 
    "Teknik Kimia", "Teknik Lingkungan", "Sistem Komputer", "Teknik Telekomunikasi",
    "Teknik Biomedis", "Teknik Geologi", "Teknik Geomatika", "Teknik Perkapalan",
    "Teknik Dirgantara", "Teknik Material", "Teknik Pertambangan", "Teknologi Informasi",
    "Rekayasa Perangkat Lunak", "Kecerdasan Buatan", "Sains Data", "Manajemen Informatika",
    # --- JURUSAN KESEHATAN & MEDIS ---
    "Kedokteran", "Kedokteran Gigi", "Kedokteran Hewan", "Keperawatan", "Kesehatan Masyarakat", 
    "Farmasi", "Kebidanan", "Gizi", "Fisioterapi", "Radiologi", "Analis Kesehatan", "Rekam Medis",
    # --- JURUSAN SOSHUM, BISNIS & HUKUM ---
    "Psikologi", "Hukum", "Ilmu Komunikasi", "Hubungan Internasional", "Kriminologi",
    "Ilmu Politik", "Administrasi Publik", "Administrasi Bisnis", "Manajemen", 
    "Akuntansi", "Ekonomi Pembangunan", "Ilmu Pemerintahan", "Ilmu Administrasi Negara",
    "Ilmu Administrasi Niaga", "Kesejahteraan Sosial", "Sosiologi", "Antropologi",
    "Bisnis Digital", "Kewirausahaan", "Ekonomi Syariah", "Perbankan Syariah",
    # --- JURUSAN SASTRA, BUDAYA & DESAIN ---
    "Sastra Inggris", "Sastra Indonesia", "Sastra Jepang", "Sastra Arab", "Sastra Cina",
    "Sastra Korea", "Sastra Perancis", "Sastra Jerman", "Sastra Belanda", "Ilmu Sejarah",
    "Filsafat", "Desain Komunikasi Visual", "DKV", "Desain Interior", "Desain Produk",
    "Seni Rupa", "Seni Musik", "Seni Tari", "Seni Karawitan", "Televisi dan Film",
    # --- JURUSAN PENDIDIKAN ---
    "Pendidikan Dokter", "Pendidikan Guru Sekolah Dasar", "PGSD", "PAUD",
    "Pendidikan Bahasa Inggris", "Pendidikan Bahasa Indonesia", "Pendidikan Matematika", 
    "Pendidikan Biologi", "Pendidikan Fisika", "Pendidikan Kimia", "Pendidikan Ekonomi",
    "Pendidikan Sejarah", "Pendidikan Geografi", "Pendidikan Sosiologi", "Bimbingan Konseling",
    "Pendidikan Agama Islam", "Manajemen Pendidikan", "Teknologi Pendidikan",
    # --- JURUSAN AGRO & SAINS TERAPAN ---
    "Agribisnis", "Agroteknologi", "Kehutanan", "Peternakan", "Ilmu Kelautan", "Oseanografi",
    "Perikanan", "Pariwisata", "Ilmu Perpustakaan", "Jurnalistik", "Broadcasting", 
    "Hubungan Masyarakat", "Matematika", "Fisika", "Kimia", "Biologi", "Statistika", "Aktuaria"
]

KEYWORD_TEMPLATES = [
    '"{jurusan}" "{universitas}" "{tahun}" chat.whatsapp.com',
    'grup whatsapp maba "{jurusan}" "{universitas}" {tahun}',
    'link grup wa "{jurusan}" "{universitas}" angkatan {tahun}',
    'grup wa maba "{universitas}" {tahun} chat.whatsapp.com',
    'grup whatsapp mahasiswa baru "{universitas}" {tahun}',
    'link grup whatsapp "{jurusan}" {tahun} chat.whatsapp.com'
]

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0'
]

OUTPUT_DIR = "output"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "wahub_links_valid.csv")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")

# Pastikan folder output ada
os.makedirs(OUTPUT_DIR, exist_ok=True)

# State untuk sistem anti-blokir
state = {
    "ddg_blocked_count": 0,
    "ddg_disabled": False,
    "blocked_queries_ddg": 0
}

# Fungsi Checkpoint untuk Resume
def load_checkpoint() -> int:
    if os.path.exists(CHECKPOINT_FILE):
        try:
            import json
            with open(CHECKPOINT_FILE, 'r') as f:
                data = json.load(f)
                idx = data.get("last_index", 0)
                if idx > 0:
                    print(f"[*] Melanjutkan dari checkpoint: Query Index ke-{idx}")
                return idx
        except Exception as e:
            print(f"[!] Gagal membaca checkpoint: {e}")
    return 0

def save_checkpoint(index: int):
    try:
        import json
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({"last_index": index}, f)
    except Exception as e:
        print(f"[!] Gagal menyimpan checkpoint: {e}")

import subprocess
import socket

def setup_multiplex_tor():
    if not USE_TOR_PROXY: return
    print(f"[*] Menyiapkan {NUM_TOR_INSTANCES} instance Tor Multiplexing...")
    
    os.makedirs("tor_data", exist_ok=True)
    subprocess.run(["killall", "tor"], capture_output=True) # Bersihkan tor lama
    
    for p in proxy_pool:
        socks_port = p["socks_port"]
        control_port = p["control_port"]
        data_dir = f"tor_data/tor{socks_port}"
        os.makedirs(data_dir, exist_ok=True)
        
        torrc_content = f"SocksPort {socks_port}\nControlPort {control_port}\nDataDirectory {data_dir}\nCookieAuthentication 0\n"
        torrc_path = f"{data_dir}/torrc"
        with open(torrc_path, "w") as f:
            f.write(torrc_content)
            
        subprocess.Popen(["tor", "-f", torrc_path], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    print("[*] Menunggu 10 detik agar seluruh instance Tor tersambung...")
    time.sleep(10)
    print("[+] Tor Multiplexing siap!")

def rotate_tor_ip_fast(proxy_info) -> bool:
    control_port = proxy_info["control_port"]
    socks_port = proxy_info["socks_port"]
    
    if not proxy_info["lock"].acquire(blocking=False):
        proxy_info["lock"].acquire(blocking=True)
        proxy_info["lock"].release()
        return True
        
    try:
        print(f"[*] Mengirim sinyal NEWNYM ke Tor Port {socks_port} (Control {control_port})...")
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.connect(("127.0.0.1", control_port))
        s.sendall(b"AUTHENTICATE \"\"\r\n")
        auth_resp = s.recv(1024)
        if b"250" not in auth_resp:
            print(f"    [!] Gagal auth: {auth_resp}")
            return False
            
        s.sendall(b"SIGNAL NEWNYM\r\n")
        sig_resp = s.recv(1024)
        s.close()
        
        if b"250" in sig_resp:
            time.sleep(2) # Jeda super singkat
            print(f"    [+] IP untuk port {socks_port} berhasil diputar!")
            return True
        return False
    except Exception as e:
        print(f"    [!] Gagal NEWNYM port {socks_port}: {e}")
        return False
    finally:
        proxy_info["lock"].release()

## Anti-Block & Validasi Response
Deteksi respon yang mengindikasikan CAPTCHA atau blokir (HTTP 429/503).

In [50]:
def is_blocked_response(response_text: str, status_code: int) -> bool:
    if status_code in [403, 429, 503]:
        return True
    
    text_lower = response_text.lower()
    keywords = ["unusual traffic", "captcha", "recaptcha", "our systems have detected"]
    for kw in keywords:
        if kw in text_lower:
            return True
            
    return False

## Query Builder & Data Cleaning
Fungsi untuk membersihkan data perguruan tinggi dan menghasilkan kombinasi query.

In [51]:
def load_and_clean_universities(csv_path="/content/sample_data/kuliah.csv") -> List[str]:
    import os
    valid_path = csv_path
    possible_paths = [
        csv_path,
        "perguruan-tinggi.csv",
        os.path.join(os.getcwd(), "perguruan-tinggi.csv"),
        os.path.join(os.getcwd(), 'scrapgrub', "perguruan-tinggi.csv"),
        r"c:\Users\Dragon\Music\scrapgrub\perguruan-tinggi.csv",
        "/mnt/c/Users/Dragon/Music/scrapgrub/perguruan-tinggi.csv"
    ]
    try:
        possible_paths.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), csv_path))
    except NameError:
        pass
    for p in possible_paths:
        if os.path.exists(p):
            valid_path = p
            break
    print(f"[*] Melakukan data cleaning pada {valid_path}...")
    try:
        df = pd.read_csv(valid_path)
        print("[+] CSV berhasil dimuat. Preview 5 baris pertama:")
        print(df.head())
        if 'Nama' in df.columns:
            # 1. Pastikan string, hilangkan whitespace berlebih, dan hapus duplikat
            namas = df['Nama'].dropna().astype(str).str.strip()
            
            # 2. Filter: Ambil hanya yang bentuknya masuk akal (panjang karakter > 3)
            namas = namas[namas.str.len() > 3]
            
            # 3. Filter: Akurasi (hanya kata kunci yang relevan sebagai perguruan tinggi)
            kata_kunci = ['universitas', 'institut', 'politeknik', 'sekolah tinggi', 'akademi', 'uin', 'iain', 'stie', 'stmik']
            mask = namas.str.lower().apply(lambda x: any(k in x for k in kata_kunci))
            
            namas_bersih = namas[mask].drop_duplicates().tolist()
            
            # Jika hasil filter sangat sedikit (salah format?), kembalikan ke awal tanpa filter kata kunci
            if len(namas_bersih) < 100:
                namas_bersih = namas.drop_duplicates().tolist()
                
            print(f"[+] Data cleaning selesai. Didapatkan {len(namas_bersih)} institusi dari CSV.")
            return namas_bersih
        
        print("[!] Kolom 'Nama' tidak ditemukan di CSV. Menggunakan data fallback.")
        return DAFTAR_UNIVERSITAS_FALLBACK
    except Exception as e:
        print(f"[!] Gagal load atau clean CSV ({e}). Menggunakan data fallback.")
        return DAFTAR_UNIVERSITAS_FALLBACK

def build_queries() -> List[Tuple[str, str, int, str]]:
    universities = load_and_clean_universities()
    queries = []
    
    # Kita menggunakan format Tuple untuk menghemat memori (RAM)
    # karena kombinasi bisa mencapai jutaan baris.
    for univ in universities:
        for jurusan in DAFTAR_FAKULTAS_JURUSAN:
            for tahun in TAHUN_RANGE:
                for template in KEYWORD_TEMPLATES:
                    queries.append((univ, jurusan, tahun, template))
    
    # Tetapkan seed yang deterministik agar hasil acak selalu persis sama antar sesi eksekusi,
    # sehingga saat skrip dilanjutkan dari checkpoint, ia berada pada urutan yang tepat.
    random.seed(42)
    random.shuffle(queries)
    random.seed() # reset seed
    return queries

## Source B - DuckDuckGo Search
Scraping DuckDuckGo versi HTML untuk mendapat hasil pencarian tambahan secara gratis.

In [53]:
def search_duckduckgo(query_obj: Dict, max_results: int = 10) -> List[Dict]:
    if state["ddg_disabled"]:
        return []
        
    query_str = query_obj["query"]
    print(f"[*] Mencari di DuckDuckGo: {query_str}")
    results = []
    
    # Rotasi User-Agent
    headers = {
        'User-Agent': random.choice(USER_AGENTS)
    }
    
    url = "https://html.duckduckgo.com/html/"
    data = {'q': query_str}
    
    proxy = random.choice(proxy_pool) if USE_TOR_PROXY else None
    proxies = {"http": f"socks5h://127.0.0.1:{proxy['socks_port']}", "https": f"socks5h://127.0.0.1:{proxy['socks_port']}"} if proxy else None
    
    try:
        time.sleep(random.uniform(0.1, 0.5))
        response = http_session.post(url, headers=headers, data=data, proxies=proxies, timeout=15)
        
        # Cek apakah diblokir
        if is_blocked_response(response.text, response.status_code):
            print(f"[!] DuckDuckGo memblokir query ini (terdeteksi rate-limit).")
            state["ddg_blocked_count"] += 1
            state["blocked_queries_ddg"] += 1
            
            # Rotasi Cepat via ControlPort
            if USE_TOR_PROXY and rotate_tor_ip_fast(proxy):
                pass
            
            # Jangan matikan DDG, karena ini satu-satunya search engine kita
            # Kita andalkan instance lain yang IP nya masih bersih
            return []
            
        response.raise_for_status()
        state["ddg_blocked_count"] = 0 # reset on success
        
        soup = BeautifulSoup(response.text, 'html.parser')
        for a in soup.find_all('a', class_='result__url', limit=max_results):
            link = a.get('href')
            if link and 'uddg=' in link:
                import urllib.parse
                parsed = urllib.parse.urlparse(link)
                qs = urllib.parse.parse_qs(parsed.query)
                if 'uddg' in qs:
                    results.append({
                        "url": qs['uddg'][0].strip(),
                        "source": "duckduckgo",
                        "meta": query_obj
                    })
            elif link:
                results.append({
                    "url": link.strip(),
                    "source": "duckduckgo",
                    "meta": query_obj
                })
    except Exception as e:
        print(f"[!] Error saat DuckDuckGo search untuk '{query_str}': {e}")
        # Jika error koneksi (timeout/unreachable), berarti node Tor ini mati. Putar IP-nya!
        if USE_TOR_PROXY and proxy:
            rotate_tor_ip_fast(proxy)
        
    return results

## Ekstraksi Link & Deduplikasi
Fungsi untuk mengekstrak link whatsapp dari halaman web dan menghilangkan duplikat.

In [57]:
def extract_wa_links_from_url(page_url: str, source_info: Dict) -> List[Dict]:
    pass # Digantikan oleh versi async di bawah

def process_image_for_wa_link(image_bytes: bytes) -> List[str]:
    kodes = []
    pattern = r'chat\.whatsapp\.com/(?:invite/)?([A-Za-z0-9]{18,24})'
    try:
        img = Image.open(io.BytesIO(image_bytes))
        
        if qr_decode:
            decoded_objects = qr_decode(img)
            for obj in decoded_objects:
                data = obj.data.decode("utf-8", errors="ignore")
                matches = re.finditer(pattern, data)
                for match in matches:
                    kodes.append(match.group(1))
                    
        if pytesseract and not kodes:
            text = pytesseract.image_to_string(img)
            matches = re.finditer(pattern, text)
            for match in matches:
                kodes.append(match.group(1))
    except Exception as e:
        pass
    return kodes

async def extract_wa_links_from_url_async(session: aiohttp.ClientSession, page_url: str, source_info: Dict) -> List[Dict]:
    kandidat = []
    try:
        await asyncio.sleep(random.uniform(0.1, 0.5)) # Mencegah spam instan
        async with session.get(page_url, timeout=10) as response:
            if response.status == 200 and 'text/html' in response.headers.get('Content-Type', ''):
                html_content = await response.text()
                
                # Regex untuk match link invite WA
                pattern = r'chat\.whatsapp\.com/(?:invite/)?([A-Za-z0-9]{18,24})'
                matches = re.finditer(pattern, html_content)
                
                for match in matches:
                    kode_invite = match.group(1)
                    kanonik_url = f"https://chat.whatsapp.com/{kode_invite}"
                    kandidat.append({
                        "link": kanonik_url, "kode_invite": kode_invite,
                        "meta": source_info["meta"], "sumber": source_info["source"]
                    })
                    
                # Ekstraksi dari gambar (poster/brosur)
                import urllib.parse
                soup = BeautifulSoup(html_content, 'html.parser')
                img_tags = soup.find_all('img')
                for img in img_tags:
                    src = img.get('src')
                    alt = img.get('alt', '').lower()
                    if not src: continue
                    
                    # Filter cerdas: hanya unduh gambar yang kemungkinan poster
                    src_lower = src.lower()
                    keywords = ['poster', 'brosur', 'pamflet', 'qr', 'barcode', 'wa', 'whatsapp', 'group', 'grup', 'join', 'maba', 'mahasiswa']
                    is_poster = any(k in src_lower or k in alt for k in keywords)
                    
                    if is_poster:
                        img_url = src if src.startswith('http') else urllib.parse.urljoin(page_url, src)
                        try:
                            async with session.get(img_url, timeout=10) as img_res:
                                if img_res.status == 200:
                                    img_bytes = await img_res.read()
                                    kodes_img = await asyncio.to_thread(process_image_for_wa_link, img_bytes)
                                    for k_inv in kodes_img:
                                        kandidat.append({
                                            "link": f"https://chat.whatsapp.com/{k_inv}",
                                            "kode_invite": k_inv,
                                            "meta": source_info["meta"],
                                            "sumber": source_info["source"] + " (Image)"
                                        })
                        except Exception:
                            pass
            else:
                print(f"    [!] Skip {page_url}: HTTP {response.status}")
    except Exception as e:
        print(f"    [!] Ekstraksi gagal {page_url}: {str(e)[:50]}")
    return kandidat

def dedup_links(kandidat_baru: List[Dict], seen_codes: Set[str]) -> List[Dict]:
    unik = []
    for item in kandidat_baru:
        kode = item["kode_invite"]
        if kode not in seen_codes:
            seen_codes.add(kode)
            unik.append(item)
    return unik

## Validator (Async)
Mengecek apakah link invite WA masih valid dengan mendapatkan og:title dari halamannya secara paralel.

In [58]:
async def validate_wa_link(session: aiohttp.ClientSession, semaphore: asyncio.Semaphore, item: Dict) -> Optional[Dict]:
    url = item["link"]
    
    async with semaphore:
        try:
            # Gunakan delay kecil sebelum request untuk meratakan beban
            await asyncio.sleep(random.uniform(0.1, 0.5))
            
            async with session.get(url, timeout=10) as response:
                if response.status == 200:
                    html = await response.text()
                    soup = BeautifulSoup(html, 'html.parser')
                    
                    og_title_meta = soup.find('meta', property='og:title')
                    og_title = og_title_meta['content'] if og_title_meta else ""
                    
                    # Validasi: title harus ada dan bukan title generik WA
                    title_lower = og_title.strip().lower()
                    if og_title and not title_lower in ["whatsapp group invite", "join chat on whatsapp"]:
                        item["nama_grup"] = og_title.strip()
                        return item
        except Exception as e:
            pass
            
    return None

async def validate_batch_async(batch_kandidat: List[Dict]) -> List[Dict]:
    valid_links = []
    semaphore = asyncio.Semaphore(50) # Maksimal 10 request bersamaan
    
    headers = {
        'User-Agent': random.choice(USER_AGENTS)
    }
    
    proxy_url = os.environ.get('http_proxy', 'socks5://127.0.0.1:9050').replace('socks5h://', 'socks5://')
    connector = ProxyConnector.from_url(proxy_url) if (USE_TOR_PROXY and ProxyConnector) else None
    async with aiohttp.ClientSession(headers=headers, connector=connector) as session:
        tasks = [validate_wa_link(session, semaphore, item) for item in batch_kandidat]
        results = await asyncio.gather(*tasks)
        
        for res in results:
            if res:
                valid_links.append(res)
                
    return valid_links

## Kategorisasi Otomatis
Menentukan universitas, jurusan, dan tahun berdasarkan nama grup atau fallback ke metadata.

In [59]:
def kategorisasi_link(item: Dict, universities: List[str]) -> Dict:
    nama_grup = item.get("nama_grup", "").lower()
    meta = item.get("meta", {})
    
    univ_match = ""
    jurusan_match = ""
    tahun_match = ""
    
    # 1. Coba match universitas dari nama grup
    for univ in universities:
        if univ.lower() in nama_grup:
            univ_match = univ
            break
            
    # 2. Coba match jurusan dari nama grup
    for jurusan in DAFTAR_FAKULTAS_JURUSAN:
        if jurusan.lower() in nama_grup:
            jurusan_match = jurusan
            break
            
    # 3. Coba match tahun dari nama grup
    for tahun in TAHUN_RANGE:
        if str(tahun) in nama_grup:
            tahun_match = tahun
            break
            
    # Fallback ke metadata jika tidak ketemu di nama grup
    hasil = item.copy()
    hasil["universitas"] = univ_match if univ_match else meta.get("universitas", "")
    hasil["fakultas_jurusan"] = jurusan_match if jurusan_match else meta.get("fakultas_jurusan", "")
    hasil["tahun"] = tahun_match if tahun_match else meta.get("tahun", "")
    hasil["tanggal_scrape"] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Hapus meta dictionary agar clean untuk CSV
    if "meta" in hasil:
        del hasil["meta"]
        
    return hasil

## Export CSV (Incremental Save)
Menyimpan data link valid ke dalam file CSV tanpa me-replace yang sudah ada (append atau create new).

In [60]:
def save_to_csv_incremental(links: List[Dict]):
    if not links:
        return
        
    file_exists = os.path.isfile(OUTPUT_FILE)
    
    df = pd.DataFrame(links)
    
    columns = ["link", "kode_invite", "nama_grup", "universitas", "fakultas_jurusan", "tahun", "sumber", "tanggal_scrape"]
    
    for col in columns:
        if col not in df.columns:
            df[col] = ""
            
    df = df[columns]
    
    df.to_csv(OUTPUT_FILE, mode='a', header=not file_exists, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"[+] Berhasil menyimpan {len(links)} link baru ke {OUTPUT_FILE}")

async def process_query_async(query_tuple, actual_index, total_queries, universities_list, seen_codes) -> int:
    univ, jurusan, tahun, template = query_tuple
    query_obj = {
        "query": template.format(jurusan=jurusan, universitas=univ, tahun=tahun),
        "universitas": univ,
        "fakultas_jurusan": jurusan,
        "tahun": tahun
    }
    
    print(f"\n--- Proses Query {actual_index+1}/{total_queries} ---")
    
    search_results = []
    
    active_engines = ["ddg"]
    if not active_engines:
        print("[!] Semua mesin pencari telah terblokir. Lewati query.")
        return 0
        
    selected_engine = active_engines[actual_index % len(active_engines)]
    
    res = []
    res = await asyncio.to_thread(search_duckduckgo, query_obj, 10)
    search_results.extend(res)
        
    if not search_results:
        return 0
        
    print(f"[*] Menemukan {len(search_results)} halaman potensial untuk query {actual_index+1}.")
    
    batch_kandidat = []
    if search_results:
        headers = {'User-Agent': random.choice(USER_AGENTS)}
        # JANGAN gunakan Tor untuk mengekstrak website target (seperti blog, web universitas, dll)
        # Website-website ini seringkali berada di belakang Cloudflare yang memblokir IP Tor secara agresif.
        # Penggunaan IP asli di sini sangat aman karena kita mendistribusikan request ke ratusan domain yang berbeda, 
        # sehingga tidak ada 1 domain yang merasa di-spam.
        connector = None 
        async with aiohttp.ClientSession(headers=headers, connector=connector) as session:
            extract_tasks = [extract_wa_links_from_url_async(session, res["url"], res) for res in search_results]
            extracted_results = await asyncio.gather(*extract_tasks)
            for k_list in extracted_results:
                batch_kandidat.extend(k_list)
        
    unik_kandidat = dedup_links(batch_kandidat, seen_codes)
    print(f"[*] Dari {len(batch_kandidat)} link diekstrak, {len(unik_kandidat)} kandidat unik baru untuk query {actual_index+1}.")
    
    if not unik_kandidat:
        return 0
        
    print(f"[*] Memvalidasi {len(unik_kandidat)} kandidat secara paralel untuk query {actual_index+1}...")
    valid_links_batch = await validate_batch_async(unik_kandidat)
    print(f"[*] Hasil validasi query {actual_index+1}: {len(valid_links_batch)} link valid.")
    
    if valid_links_batch:
        kategori_batch = [kategorisasi_link(item, universities_list) for item in valid_links_batch]
        save_to_csv_incremental(kategori_batch)
        
        print(f"\n[+] MENDAPATKAN {len(kategori_batch)} LINK VALID BARU DARI QUERY {actual_index+1}:")
        for item in kategori_batch:
            print(f"    -> {item['nama_grup']}")
            print(f"       {item['universitas']} - {item['fakultas_jurusan']} ({item['tahun']})")
            print(f"       Link: {item['link']}")
        print("")
        
        return len(kategori_batch)
    return 0

## Eksekusi Utama
Menyatukan seluruh alur dari pencarian sampai ekstraksi, validasi, dan penyimpanan.

In [61]:
async def main_loop():
    target_str = "Tanpa Batas" if TARGET_VALID_LINKS == float('inf') else str(TARGET_VALID_LINKS)
    print(f"=== Mulai Scraping WAHub (Target: {target_str} valid links) ===")
    
    setup_multiplex_tor()
    start_time = time.time()
    
    queries_tuples = build_queries()
    print(f"[*] Total kombinasi query yang di-generate: {len(queries_tuples)}")
    
    # Kita ambil state list universitas yang digunakan saat ini untuk fungsi kategorisasi
    universities_list = load_and_clean_universities()
    
    seen_codes = set()
    total_valid = 0
    total_kandidat_ditemukan = 0
    
    if os.path.exists(OUTPUT_FILE):
        try:
            df_existing = pd.read_csv(OUTPUT_FILE)
            if "kode_invite" in df_existing.columns:
                seen_codes.update(df_existing["kode_invite"].astype(str).tolist())
                total_valid = len(seen_codes)
                print(f"[*] Ditemukan {total_valid} link valid di sesi sebelumnya.")
        except Exception as e:
            print(f"[!] Gagal membaca CSV lama: {e}")
            
    if total_valid >= TARGET_VALID_LINKS:
        print(f"[!] Target {TARGET_VALID_LINKS} link sudah tercapai di awal.")
        return
        
    start_index = load_checkpoint()

    for batch_start in range(start_index, len(queries_tuples), CONCURRENT_WORKERS):
        if total_valid >= TARGET_VALID_LINKS:
            break
            
        if state["ddg_disabled"]:
            print("[!] FATAL: DuckDuckGo telah memblokir sepenuhnya. Menghentikan script.")
            break
            
        chunk = queries_tuples[batch_start:batch_start + CONCURRENT_WORKERS]
        tasks = []
        for i_offset, query_tuple in enumerate(chunk):
            actual_index = batch_start + i_offset
            tasks.append(process_query_async(query_tuple, actual_index, len(queries_tuples), universities_list, seen_codes))
            
        # Mengeksekusi batch secara paralel
        results = await asyncio.gather(*tasks)
        total_valid += sum(results)
        
        print(f"=== PROGRESS BATCH: {total_valid} valid links (Target: {target_str}) ===")
        
        # Simpan state checkpoint setelah 1 batch selesai
        save_checkpoint(batch_start + len(chunk))
        
    elapsed = time.time() - start_time
    print(f"\n=== SUMMARY EKSEKUSI ===")
    print(f"Waktu eksekusi: {elapsed/60:.2f} menit")
    print(f"Total halaman kandidat dikunjungi: {total_kandidat_ditemukan}")
    print(f"Total link valid unik dikumpulkan: {total_valid}")

    active_engine_list = ["google", "ddg", "yahoo", "gibiru", "searxng"]
    for engine in active_engine_list:
        print(f"Query diblokir oleh {engine.capitalize()}: {state[f'blocked_queries_{engine}']}")
    for engine in active_engine_list:
        if state[f"{engine}_disabled"]:
            print(f"[!] {engine.capitalize()} dinonaktifkan akibat pemblokiran beruntun.")

    print("\n=== AI NEXT STEP RECOMMENDATION ===")
    if not any(not state[f"{e}_disabled"] for e in active_engine_list):
        print("[AI-SUGGESTION] SEMUA mesin pencari memblokir IP ini secara bersamaan. Hentikan eksekusi, ganti VPN/IP, gunakan IP Residential, atau beli akses API pencarian resmi.")
    elif total_valid < 5 and any(state[f"blocked_queries_{e}"] > 10 for e in active_engine_list):
        print("[AI-SUGGESTION] Banyak pemblokiran dan yield sangat rendah. Skema delay dan rotasi User-Agent kurang efektif di environment ini. Butuh evaluasi sistem penyamaran.")
    elif total_valid >= TARGET_VALID_LINKS:
        print("[AI-SUGGESTION] Target link berhasil tercapai. Langkah selanjutnya adalah memproses/format output CSV jika diperlukan untuk database.")
    else:
        print("[AI-SUGGESTION] Proses scraping selesai atau terhenti. Jika target belum tercapai dan ingin melacak sisa query, Anda cukup jalankan ulang script ini dari checkpoint.")

In [ ]:
if __name__ == "__main__":
    try:
        # Menangani issue event loop jika dijalankan di dalam Jupyter Notebook
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            loop = None
            
        if loop and loop.is_running():
            print("[*] Mendeteksi Jupyter Notebook / Event loop yang sudah berjalan. Menerapkan nest_asyncio...")
            import nest_asyncio
            nest_asyncio.apply()
            
        asyncio.run(main_loop())
    except KeyboardInterrupt:
        print("\n[!] Dihentikan paksa oleh pengguna.")

[*] Mendeteksi Jupyter Notebook / Event loop yang sudah berjalan. Menerapkan nest_asyncio...
=== Mulai Scraping WAHub (Target: Tanpa Batas valid links) ===
[*] Melakukan data cleaning pada /content/sample_data/kuliah.csv...
[+] CSV berhasil dimuat. Preview 5 baris pertama:
   No                                      Nama  LLDikti Unnamed: 3
0   1  AKADEMI ADMINISTRASI RUMAH SAKIT MATARAM      8.0        NaN
1   2    Akademi Akuntansi (AKTAN) Boekittinggi     10.0        NaN
2   3                 Akademi Akuntansi Bandung      4.0        NaN
3   4             Akademi Akuntansi Bina Insani      4.0        NaN
4   5               Akademi Akuntansi Borobudur      3.0        NaN
[+] Data cleaning selesai. Didapatkan 4411 institusi dari CSV.
[*] Total kombinasi query yang di-generate: 14503368
[*] Melakukan data cleaning pada /content/sample_data/kuliah.csv...
[+] CSV berhasil dimuat. Preview 5 baris pertama:
   No                                      Nama  LLDikti Unnamed: 3
0   1  AKADEMI A